# Output Parsers Lab

This notebook demonstrates different output parsers in LangChain for structured data extraction.


In [4]:
from langchain.chat_models import init_chat_model

from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate

llm = init_chat_model("gpt-4o-mini", model_provider="openai")


## String Parser (Most Common)


In [5]:
prompt = PromptTemplate(
    input_variables=["question"],
    template="You are a helpful assistant. Answer this question: {question}?"
)

from langchain_core.output_parsers import StrOutputParser
# Clean string output
string_parser = StrOutputParser()

# In a chain
chain = prompt | llm | string_parser
result = chain.invoke({"question": "What is Python?"})
print("Result :", result)


Result : Python is a high-level, interpreted programming language known for its simplicity and readability, making it an excellent choice for beginners and experienced programmers alike. It was created by Guido van Rossum and first released in 1991. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming.

Key features of Python include:

1. **Easy to Learn and Use**: It has a clear and straightforward syntax that enhances readability, making it accessible for newcomers.

2. **Extensive Libraries and Frameworks**: Python boasts a rich ecosystem of libraries and frameworks, such as NumPy for numerical computing, Pandas for data analysis, and Django and Flask for web development, among others.

3. **Cross-Platform Compatibility**: Python can run on various operating systems, including Windows, macOS, and Linux, allowing for greater flexibility in development.

4. **Dynamic Typing**: Variables in Python do not require explicit decl

## List Parser


In [8]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# List parser
#creating the instance of the class CommaSeparatedListOutputParser
list_parser = CommaSeparatedListOutputParser()


print("Instructions :",list_parser.get_format_instructions())
# # Create prompt with format instructions
list_prompt = PromptTemplate(
    template="give top 5 programming languages:\n{format_instructions}",
    input_variables=[],
    partial_variables={"format_instructions": list_parser.get_format_instructions()}
)

chain = list_prompt | llm | list_parser
result = chain.invoke({})
print("Result :", result)
print("Result :", result[0])


Instructions : Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`
Result : ['Python', 'JavaScript', 'Java', 'C#', 'C++']
Result : Python


## JSON Parser


In [9]:
from langchain_core.output_parsers import JsonOutputParser
# JSON parser for flexible structured data
json_parser = JsonOutputParser()
print("Instructions :",json_parser.get_format_instructions())
json_prompt = ChatPromptTemplate.from_template(
    """Analyze this product and return a JSON object with these fields:
    - name: product name
    - price: estimated price in USD
    - category: product category
    - features: array of key features
    
    Product: {product}
    
    {format_instructions}"""
).partial(format_instructions=json_parser.get_format_instructions())

json_chain = json_prompt | llm | json_parser
result = json_chain.invoke({"product": "iPhone 15 Pro"})
print("Result :", result)
print("Price :", result["price"])


Instructions : Return a JSON object.
Result : {'name': 'iPhone 15 Pro', 'price': 999, 'category': 'Smartphone', 'features': ['A17 Pro chip for high performance', '6.1-inch Super Retina XDR display', 'Triple-camera system with ProRAW and ProRes video', '5G capable', 'Durable design with Ceramic Shield front', 'iOS 17 with enhanced privacy features', 'MagSafe technology for easy attachment of accessories']}
Price : 999


## Pydantic parser

In [10]:
from typing import List
from pydantic import BaseModel, Field, field_validator


from langchain_core.output_parsers import PydanticOutputParser

# 1. Define your data schema with Pydantic
class Person(BaseModel):
    name: str = Field(..., description="The person's name")
    age: int = Field(..., description="Age in years")
    hobbies: List[str] = Field(default_factory=list, description="List of hobbies")

    @field_validator("age")
    def age_must_be_positive(cls, v):
        if v < 0:
            raise ValueError("age must be non-negative")
        return v

class People(BaseModel):
    people: List[Person]

# 2. Instantiate the parser with your Pydantic model
parser = PydanticOutputParser(pydantic_object=People)

print("Instructions :",parser.get_format_instructions())
print("--------------------------------")

# 3. Build a prompt that instructs the LLM to output JSON matching your schema
prompt = PromptTemplate(
    template="""
You are given a text describing some people. Extract list of people details in JSON.
Output must follow this JSON schema exactly:
{format_instructions}

Text:
{text}
""",
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

# 4. Ask the LLM

input_text = "Alice is 30 years old and loves painting. Bob is 25 and enjoys cycling and chess."
_prompt = prompt.format_prompt(text=input_text)


chain = prompt | llm | parser
result = chain.invoke({"text": input_text})
print("Result :")
print(result)


Instructions : The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"$defs": {"Person": {"properties": {"name": {"description": "The person's name", "title": "Name", "type": "string"}, "age": {"description": "Age in years", "title": "Age", "type": "integer"}, "hobbies": {"description": "List of hobbies", "items": {"type": "string"}, "title": "Hobbies", "type": "array"}}, "required": ["name", "age"], "title": "Person", "type": "object"}}, "properties": {"people": {"items": {"$ref": "#/$defs/Person"}, "title": "People", "type": "array"}}, "required": ["people"]}
```
------------------------------

Result :
people=[Person(name='Alice', age=30, hobbies=['painting']), Person(name='Bob', age=25, hobbies=['cycling', 'chess'])]


In [12]:
print(result.people)
print("Name :", result.people[0].name)
print("Age :", result.people[0].age)
print("Hobbies :", result.people[0].hobbies)

print("--------------------------------")



[Person(name='Alice', age=30, hobbies=['painting']), Person(name='Bob', age=25, hobbies=['cycling', 'chess'])]
Name : Alice
Age : 30
Hobbies : ['painting']
--------------------------------


In [17]:
result.model_dump_json()

'{"people":[{"name":"Alice","age":30,"hobbies":["painting"]},{"name":"Bob","age":25,"hobbies":["cycling","chess"]}]}'

## Pydantic Parser Example 2: Recipe Extraction


In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field, field_validator
from langchain_core.output_parsers import PydanticOutputParser

# Define nested models for recipe
class Ingredient(BaseModel):
    name: str = Field(..., description="Ingredient name")
    quantity: str = Field(..., description="Quantity with unit (e.g., '2 cups', '500g')")
    optional: bool = Field(default=False, description="Whether ingredient is optional")

class RecipeStep(BaseModel):
    step_number: int = Field(..., description="Step number in sequence")
    instruction: str = Field(..., description="Detailed instruction for this step")
    duration_minutes: Optional[int] = Field(None, description="Time required in minutes")

class Recipe(BaseModel):
    name: str = Field(..., description="Recipe name")
    cuisine: str = Field(..., description="Type of cuisine")
    servings: int = Field(..., description="Number of servings")
    prep_time_minutes: int = Field(..., description="Preparation time in minutes")
    cook_time_minutes: int = Field(..., description="Cooking time in minutes")
    ingredients: List[Ingredient] = Field(..., description="List of ingredients")
    steps: List[RecipeStep] = Field(..., description="Cooking steps in order")
    difficulty: str = Field(..., description="Difficulty level: Easy, Medium, or Hard")
    
    @field_validator("servings")
    def servings_must_be_positive(cls, v):
        if v <= 0:
            raise ValueError("servings must be positive")
        return v
    
    @field_validator("difficulty")
    def difficulty_must_be_valid(cls, v):
        valid_levels = ["Easy", "Medium", "Hard"]
        if v not in valid_levels:
            raise ValueError(f"difficulty must be one of {valid_levels}")
        return v

# Create parser
recipe_parser = PydanticOutputParser(pydantic_object=Recipe)

# Create prompt
recipe_prompt = PromptTemplate(
    template="""
Give me a complete recipe for {recipe_topic}, and format it as JSON.
Output must follow this JSON schema exactly:
{format_instructions}
""",
    input_variables=["recipe_topic"],
    partial_variables={"format_instructions": recipe_parser.get_format_instructions()}
)

# Example usage
recipe_topic = "Chocolate Chip Cookies"

recipe_chain = recipe_prompt | llm | recipe_parser
recipe_result = recipe_chain.invoke({"recipe_topic": recipe_topic})

print("Recipe Name:", recipe_result.name)
print("Cuisine:", recipe_result.cuisine)
print("Servings:", recipe_result.servings)
print("Total Time:", recipe_result.prep_time_minutes + recipe_result.cook_time_minutes, "minutes")
print("\nIngredients:")
for ing in recipe_result.ingredients:
    optional_text = " (optional)" if ing.optional else ""
    print(f"  - {ing.quantity} {ing.name}{optional_text}")
print("\nFirst Step:", recipe_result.steps[0].instruction)
print("Difficulty:", recipe_result.difficulty)


Recipe Name: Chocolate Chip Cookies
Cuisine: American
Servings: 24
Total Time: 27 minutes

Ingredients:
  - 2 cups all-purpose flour
  - 1 cup (softened) butter
  - 3/4 cup brown sugar
  - 3/4 cup white sugar
  - 2 eggs
  - 1 tsp vanilla extract (optional)
  - 2 cups chocolate chips

First Step: Preheat oven to 375°F (190°C).
Difficulty: Easy
